# 9.12 為什麼模型必須讀候選的意思？


上一節的正確答案排在第一列。如果模型只記「卡片問題選第1個」，把選項換位便會選錯。要得到可使用的判斷，我們需要它比較**材料與候選意義是否相符**，而非利用選項位置偷猜答案。

一個容易核對的設計，是每個候選都經過同一個評分函數 `score(state, question, meaning)`。輸入裡沒有「第幾個候選」的特徵；評分各候選時也不讀其他候選的排列。訓練時，所有分數一起算 softmax 與交叉熵代價。推論時交換候選，只會交換相應分數和機率的位置；最高分唯一時，所選的候選意義應保持一致。若並列最高，還需不依列表位置的平手規則，例如按候選固定的識別碼排序。若還加入選項位置或跨候選互動，就要重新實驗，不能沿用這個結論。

下面沿用9.11的人工分數示範這項排列關係。分數用字典按候選意義存放，相當於分數只跟著意義走、不看位置；它沒有讀懂中文，也不是模型成果，只讓我們檢查「名稱與機率一起搬動」。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

scores = {"查卡片進度": 2.0, "更改地址": -1.0, "以上皆不適合": 0.0}
forward = list(scores)
backward = list(reversed(forward))
for order in [forward, backward]:
    p = torch.tensor([scores[name] for name in order]).softmax(-1)
    print("順序", order, "機率", [round(x, 3) for x in p.tolist()], "選擇", order[int(p.argmax())])

`list(scores)`取出字典的鍵，依寫入順序排成清單`forward`；`reversed(forward)`從最後一項往前取，`list(...)`再把它排成倒序的清單`backward`。迴圈對兩種順序各做一次：`[scores[name] for name in order]`按目前順序查出分數，softmax轉成機率，`[round(x, 3) for x in p.tolist()]`把機率四捨五入到三位方便閱讀；`order[int(p.argmax())]`取出最高機率所在位置的候選名稱。

第一行輸出就是9.11的0.844、0.042、0.114；倒序後機率清單變成0.114、0.042、0.844，數字跟著名稱一起搬動，兩次都選「查卡片進度」。這是介面應有的檢查，不代表分數判斷正確；共享評分器也可能穩定地選錯。正式評估還要把 state 固定、question 改掉，例如從「下一步服務」改成「已知哪種資訊」，確認答案真的跟問題變化。

還有一個常見陷阱：使用者要重設 App 密碼，候選卻只有卡片與地址。softmax 仍會把機率加成1；它沒有自動創造「都不適合」。因此資料與介面要明列這個候選，並提供真的沒有合適答案的訓練例。只把最佳選項刪掉、仍用原本標籤算分，測到的便不是原定任務。

候選集合改變時，分母也改變。同一候選的機率可能上升或下降，不能把這個條件機率當作不隨候選改變的客觀可信度。加入兩個同義選項還可能分散機率；正式資料應避免含義重疊，或另定多答案評分規則。

練習：假設訓練資料只記正確答案的位置`label = 0`，也就是第一個候選。在程式最後加一行`print(backward[0])`，先預測會印出「以上皆不適合」，再執行核對。這表示把候選換成倒序卻不搬`label`，就等於把錯的答案教給模型。重排時要一起更新位置，並保存每個候選固定的意義ID，才有辦法比對重排前後的答案。
